# Практика №2. Полносвязная нейронная сеть (FCN / MLP)

**Задача:** классифицировать сорт риса (`Cammeo` или `Osmancik`) по морфологическим характеристикам зерна.

**Датасет:** UCI Rice (Cammeo and Osmancik), 3 810 наблюдений, 7 числовых признаков и 2 класса. Признаки получены при обработке изображений зёрен риса. Источник: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/545/rice), DOI: `10.24432/C5MW4Z`.

В этой тетради реализуется неглубокая полносвязная нейронная сеть. Каждый нейрон слоя связан со всеми выходами предыдущего слоя. Такой вариант подходит для табличных данных, представленных в виде вектора признаков.

Последовательность работы: загрузка данных, проверка качества, EDA, разделение выборки, масштабирование, проектирование FCN, обучение, оценка и сохранение модели.

In [ ]:
%pip install -q tensorflow ucimlrepo pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from ucimlrepo import fetch_ucirepo

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)

## 1. Загрузка данных

Датасет загружается из UCI с помощью `ucimlrepo`. Входные признаки содержат геометрические характеристики зёрен: площадь, периметр, длины осей, эксцентриситет, выпуклую площадь и протяжённость. Целевая переменная — сорт риса.

In [ ]:
dataset = fetch_ucirepo(id=545)
X_raw = dataset.data.features.copy()
y_raw = dataset.data.targets.iloc[:, 0].copy()

print(f"Размер матрицы признаков: {X_raw.shape}")
print("Признаки:", X_raw.columns.tolist())
print("Целевая переменная:", dataset.data.targets.columns[0])
print("Классы:", sorted(pd.Series(y_raw).dropna().astype(str).unique()))
display(X_raw.head())
display(y_raw.value_counts(dropna=False).to_frame("count"))

## 2. Проверка качества и очистка данных

Проверяем пропуски, бесконечные значения, полные дубликаты и потенциальные выбросы по правилу межквартильного размаха (IQR). Выбросы не удаляются автоматически: необычная форма зерна может быть корректным наблюдением. Строки с отсутствующей целевой переменной и полные дубликаты удаляются. Пропуски в признаках, если они встретятся, будут заполнены медианой, рассчитанной только на обучающей выборке.

In [ ]:
df = X_raw.copy()
df = df.apply(pd.to_numeric, errors="coerce")
df["target"] = y_raw.astype("object").astype(str)
df = df.replace([np.inf, -np.inf], np.nan)
df.loc[df["target"].isin(["nan", "None", ""]), "target"] = np.nan

print("Пропуски до очистки:")
display(df.isna().sum().to_frame("missing"))
print("Количество полных дубликатов:", int(df.duplicated().sum()))

rows_before = len(df)
df = df.dropna(subset=["target"]).drop_duplicates().reset_index(drop=True)
print(f"Удалено строк с отсутствующей целевой переменной/полных дубликатов: {rows_before - len(df)}")

feature_names = [column for column in df.columns if column != "target"]
X = df[feature_names].copy()
y_text = df["target"].astype(str)

print("Пропуски в признаках после проверки:")
display(X.isna().sum().to_frame("missing"))

def count_iqr_outliers(frame):
    counts = {}
    for column in frame.select_dtypes(include=np.number).columns:
        values = frame[column].dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        counts[column] = int(((values < lower) | (values > upper)).sum())
    return pd.Series(counts, name="iqr_outliers").sort_values(ascending=False)

print("Потенциальные выбросы по IQR (не удаляются автоматически):")
display(count_iqr_outliers(X).to_frame())
display(df.describe(include="all").T)

## 3. Разведочный анализ данных (EDA)

Построим распределения числовых признаков и распределение классов. Корреляционная матрица рассчитывается между семью числовыми признаками и помогает увидеть их линейные взаимосвязи. Высокая корреляция не означает причинно-следственную связь.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for column, ax in zip(feature_names, axes.ravel()):
    sns.histplot(data=df, x=column, kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Распределение: {column}")
for ax in axes.ravel()[len(feature_names):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="target", order=sorted(df["target"].unique()))
plt.title("Распределение целевых классов")
plt.xlabel("Сорт риса")
plt.ylabel("Количество наблюдений")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляционная матрица признаков")
plt.tight_layout()
plt.show()

## 4. Разделение выборки и масштабирование

Используем стратифицированное разделение примерно в пропорции 70% / 15% / 15%. Стратификация сохраняет соотношение классов в каждой части. Числовые признаки стандартизируются: из каждого значения вычитается среднее обучающей выборки и выполняется деление на стандартное отклонение. Параметры масштабирования рассчитываются только по `train`, чтобы избежать утечки данных.

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train: {len(X_train)} строк")
print(f"Validation: {len(X_val)} строк")
print(f"Test: {len(X_test)} строк")
print("Порядок классов:", label_encoder.classes_.tolist())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

## 5. Проектирование полносвязной нейронной сети

Входной слой получает вектор из семи стандартизированных признаков. Далее расположены два полносвязных скрытых слоя с функцией активации ReLU. Dropout случайным образом отключает часть нейронов на обучении и помогает уменьшить переобучение. Выходной нейрон с сигмоидой возвращает вероятность принадлежности ко второму классу.

Для бинарной классификации используется `binary_crossentropy`, оптимизатор Adam и метрики Accuracy, Precision, Recall и AUC.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model.summary()

## 6. Обучение модели

Модель обучается на `train`, а качество после каждой эпохи проверяется на `validation`. `EarlyStopping` останавливает обучение, если ошибка на валидационной выборке перестаёт уменьшаться, и восстанавливает веса лучшей эпохи. Тестовая выборка на этом этапе не используется.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1,
)

history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_df["loss"], label="train")
axes[0].plot(history_df["val_loss"], label="validation")
axes[0].set_title("Функция потерь")
axes[0].set_xlabel("Эпоха")
axes[0].legend()
axes[1].plot(history_df["accuracy"], label="train")
axes[1].plot(history_df["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Эпоха")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. Итоговая оценка на тестовой выборке

После выбора весов по `validation` один раз оцениваем модель на отложенной `test`-выборке. Помимо Accuracy рассчитываются Balanced Accuracy, Precision, Recall, F1 и ROC-AUC. Матрица ошибок показывает, какие сорта риса модель путает между собой.

In [ ]:
y_test_probability = model.predict(X_test_scaled, verbose=0).ravel()
y_test_pred = (y_test_probability >= 0.5).astype(int)

test_metrics = {
    "Accuracy": float(accuracy_score(y_test, y_test_pred)),
    "Balanced Accuracy": float(balanced_accuracy_score(y_test, y_test_pred)),
    "Precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
    "Recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
    "F1": float(f1_score(y_test, y_test_pred, zero_division=0)),
    "ROC-AUC": float(roc_auc_score(y_test, y_test_probability)),
}

print("Итоговые метрики на тестовой выборке:")
display(pd.Series(test_metrics, name="value").to_frame())
print("\nОтчёт по классам:")
print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_, zero_division=0))

cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_
)
plt.title("Матрица ошибок FCN")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.tight_layout()
plt.show()

## 8. Сохранение модели и артефактов

Сохраняем обученную нейронную сеть в формате Keras, объект стандартизации и кодировщик классов. Отдельно записываем тестовые метрики и список признаков, чтобы в дальнейшем можно было воспроизвести предобработку и интерпретировать результат.

In [ ]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

model.save(artifact_dir / "fcn_rice.keras")
joblib.dump(scaler, artifact_dir / "fcn_rice_scaler.joblib")
joblib.dump(label_encoder, artifact_dir / "fcn_rice_label_encoder.joblib")

with (artifact_dir / "fcn_rice_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "test_metrics": test_metrics,
            "features": feature_names,
            "classes": label_encoder.classes_.tolist(),
            "random_state": RANDOM_STATE,
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Модель, scaler, encoder и метрики сохранены в папку artifacts/")

## 9. Вывод

В работе реализована полносвязная нейронная сеть для классификации табличных данных. После запуска тетради зафиксируйте фактические метрики на тестовой выборке и оцените матрицу ошибок. Отдельно обратите внимание на разницу между Accuracy и Balanced Accuracy: при дисбалансе классов вторая метрика даёт более объективное представление о качестве распознавания каждого класса.